# Batch running proseg

In [2]:
import sys
print(sys.executable)

/Users/christoffer/miniconda3/envs/sc/bin/python


In [4]:
import os
import subprocess

# Base directory where input files live
base_dir = '/Volumes/processing2/hm-xenium-ms/transcripts_cleaned'

# Base output directory (results grouped by slide ID)
output_root = "/Volumes/processing2/hm-xenium-ms/data/proseg"

In [5]:
! proseg --version

proseg 3.0.11


In [8]:
input_path

'/Volumes/processing2/hm-xenium-ms/transcripts_cleaned/xenium_datasets-1/output-XETG00045__0003524__active__20230510__111824/transcripts.parquet/transcripts_genes_only.parquet'

In [11]:
file

'transcripts_genes_only.parquet'

In [13]:
root

'/Volumes/processing2/hm-xenium-ms/transcripts_cleaned/xenium_datasets-1/output-XETG00045__0003524__active__20230510__111824/transcripts.parquet'

In [ ]:
import os

def extract_slide_id_from_path(path):
    parts = os.path.normpath(path).split(os.sep)

    # find Xenium folder like: output-XETG...__inactive__20230510__111824
    output_dir = next(p for p in parts if p.startswith("output-"))

    tokens = output_dir.replace("output-", "").split("__")
    # tokens: ['XETG00045','0003385','inactive','20230510','111824']
    condition = tokens[2]
    date = tokens[3]
    return f"{condition}__{date}"


for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file.endswith("transcripts_genes_only.parquet"):
            input_path = os.path.join(root, file)

            # ✅ NEW: biologically meaningful ID
            slide_id = extract_slide_id_from_path(input_path)

            base_name = os.path.splitext(os.path.splitext(file)[0])[0]
            output_path = os.path.join(output_root, slide_id, base_name)

            # ✅ define done_flag correctly
            done_flag = os.path.join(output_path, "proseg-output.zarr")

            if os.path.isdir(done_flag):
                print(f"⏭️  Already processed → {slide_id}")
                continue

            os.makedirs(output_path, exist_ok=True)

            print(f"▶️ Running proseg on: {input_path}")
            print(f"💾 Saving to:          {output_path}")

            !proseg "{input_path}" \
              --xenium \
              --output-path "{output_path}" \

            print(f"✅ Finished: {slide_id}")

▶️ Running proseg on: /Volumes/processing2/hm-xenium-ms/transcripts_cleaned/20230829__105411__20230829_Goncalo_Petra_run2/output-XETG00047__0010759__MS-C_2012-078__20230829__105447/transcripts.parquet/transcripts_genes_only.parquet
💾 Saving to:          /Volumes/processing2/hm-xenium-ms/data/proseg/MS-C_2012-078__20230829/transcripts_genes_only
Using 16 threads
Finished reading input
Read dataset:
 31917270 transcripts
   154737 cells
      266 genes
      304 fovs
00:16:06 #####------------------------------------------------------- | log-likelihood: -169191140 | assigned: 15476796 / 31917270 (48.49%) | non-background: (43.44%)                                                                          